# HadISD 2024 All-Variable Data Cleaning and Quality Assessment

This notebook updates the HadISD workflow so that the basic cleaning and quality-control logic is applied to **all variables available in the consolidated 2024 file**.

The diagnostic figures shown in the report can still focus on one representative variable, such as pressure, but the underlying QC summaries are produced for the complete set of variables:

- `PRESS` — atmospheric pressure, from `msl`
- `U10` — 10 m zonal wind component, from `u10`
- `V10` — 10 m meridional wind component, from `v10`
- `TARIA2M` — 2 m air temperature, from `t2m`

The notebook also replaces the simple latitude/longitude scatter plot with a more map-like station coverage figure using an approximate Adriatic coastline guide.

In [ ]:
# Imports and configuration
from pathlib import Path
from collections import defaultdict
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd()

# ------------------------------------------------------------------
# CSV selection
# ------------------------------------------------------------------
# Preferred file names. If none of these exists, the notebook automatically
# selects the largest CSV in this folder, excluding known summary/co-location files.
PREFERRED_CSV_NAMES = [
    "hadisd_adriatic_2024.csv",
    "HadISD_2024.csv",
    "hadisd_2024.csv",
    "hadisd_full_2024.csv",
]

CSV_PATH = None
for name in PREFERRED_CSV_NAMES:
    p = BASE_DIR / name
    if p.exists():
        CSV_PATH = p
        break

if CSV_PATH is None:
    excluded_keywords = ["colocation", "summary", "cleaned", "qc_flagged", "station_month", "max_gap"]
    candidates = [
        p for p in BASE_DIR.glob("*.csv")
        if not any(k in p.name.lower() for k in excluded_keywords)
    ]
    if not candidates:
        raise FileNotFoundError(
            "No suitable CSV file found in this folder. Put the full-year HadISD CSV in DataAggregation/HadISD."
        )
    CSV_PATH = max(candidates, key=lambda p: p.stat().st_size)

print("Selected CSV:", CSV_PATH.name)
print("File size MB:", round(CSV_PATH.stat().st_size / 1024 / 1024, 2))

# ------------------------------------------------------------------
# Output folders
# ------------------------------------------------------------------
FIG_DIR = BASE_DIR / "reports" / "figures" / "hadisd_2024_all_variables_qc"
TABLE_DIR = BASE_DIR / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV = TABLE_DIR / "hadisd_2024_cleaned_all_variables.csv"
SUMMARY_CSV = TABLE_DIR / "hadisd_2024_all_variables_cleaning_summary.csv"
VARIABLE_SUMMARY_CSV = TABLE_DIR / "hadisd_2024_variable_summary.csv"
STATION_SUMMARY_CSV = TABLE_DIR / "hadisd_2024_station_summary_all_variables.csv"
MONTHLY_SUMMARY_CSV = TABLE_DIR / "hadisd_2024_monthly_summary_all_variables.csv"
STATION_MONTH_CSV = TABLE_DIR / "hadisd_2024_station_month_availability_by_variable.csv"
MAX_GAP_CSV = TABLE_DIR / "hadisd_2024_max_temporal_gap_by_station_variable.csv"

# ------------------------------------------------------------------
# Cleaning parameters
# ------------------------------------------------------------------
CHUNKSIZE = 500_000
YEAR = 2024
EXPECTED_HOURLY_OBS = 366 * 24  # 2024 is a leap year: 8784 hours
MAX_SAMPLE_ROWS_PER_VARIABLE = 80_000

# Standard variables expected in the consolidated HadISD 2024 file.
VARIABLES = {
    "PRESS": {
        "raw_names": ["msl", "press", "pressure", "stnlp", "slp", "b10004"],
        "description": "Atmospheric pressure",
        "unit": "hPa",
        "range": (850, 1100),
    },
    "U10": {
        "raw_names": ["u10", "u", "u_wind", "wind_u"],
        "description": "10 m zonal wind component",
        "unit": "m/s",
        "range": (-80, 80),
    },
    "V10": {
        "raw_names": ["v10", "v", "v_wind", "wind_v"],
        "description": "10 m meridional wind component",
        "unit": "m/s",
        "range": (-80, 80),
    },
    "TARIA2M": {
        "raw_names": ["t2m", "2t", "taria2m", "temperature", "temperatures", "b12101"],
        "description": "2 m air temperature",
        "unit": "°C",
        "range": (-50, 60),
    },
}

# Reverse lookup for variable mapping.
RAW_TO_STANDARD = {}
for standard, meta in VARIABLES.items():
    RAW_TO_STANDARD[standard.upper()] = standard
    for raw in meta["raw_names"]:
        RAW_TO_STANDARD[str(raw).upper()] = standard

print("Figures will be saved to:", FIG_DIR)
print("Tables will be saved to:", TABLE_DIR)

## 1. Inspect the CSV schema

This step reads only a few rows to confirm the real column names and data format. It does not load the full file into memory.

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

sample_preview = pd.read_csv(CSV_PATH, nrows=10, low_memory=False)

print("Columns:")
for c in sample_preview.columns:
    print("-", c)

print("
Preview:")
display(sample_preview.head(10))

print("
Data types:")
print(sample_preview.dtypes)

## 2. Harmonise variables into a common long format

The consolidated file may appear in two possible structures:

1. **Sensor-style format**, where each row already contains a variable code, a value, a unit, a timestamp, and station coordinates.
2. **Wide format**, where variables such as `msl`, `u10`, `v10`, and `t2m` are stored as separate columns.

The functions below convert both cases into one common long-format table with the following conceptual fields: station, timestamp, variable, value, unit, and location.

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df


def find_column(df: pd.DataFrame, candidates):
    """Find a column by trying several possible names case-insensitively."""
    lookup = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lookup:
            return lookup[cand.lower()]
    return None


def map_to_standard_variable(raw_name):
    if pd.isna(raw_name):
        return None
    key = str(raw_name).strip().upper()
    return RAW_TO_STANDARD.get(key)


def convert_units(variable: str, values: pd.Series) -> pd.Series:
    """Convert values to the standard units defined in VARIABLES when needed."""
    values = pd.to_numeric(values, errors="coerce")

    if variable == "PRESS":
        # If pressure looks like Pascals, convert to hectopascals.
        # Values around 100000 indicate Pa; values around 1000 indicate hPa.
        median_val = values.dropna().median() if values.notna().any() else np.nan
        if pd.notna(median_val) and median_val > 2000:
            values = values / 100.0

    if variable == "TARIA2M":
        # If temperature looks like Kelvin, convert to Celsius.
        median_val = values.dropna().median() if values.notna().any() else np.nan
        if pd.notna(median_val) and median_val > 150:
            values = values - 273.15

    return values


def harmonise_chunk_to_long(chunk: pd.DataFrame) -> pd.DataFrame:
    """Convert one chunk to the standard long-format HadISD table."""
    chunk = normalize_columns(chunk)

    station_col = find_column(chunk, ["station_id", "station", "id_station", "station_name", "name"])
    dt_col = find_column(chunk, ["dt", "datetime", "timestamp", "time", "date"])
    lat_col = find_column(chunk, ["lat", "latitude"])
    lon_col = find_column(chunk, ["lon", "longitude"])
    value_col = find_column(chunk, ["value", "measurement", "val"])
    sensor_code_col = find_column(chunk, ["sensor_code", "variable", "var", "code", "parameter"])
    unit_col = find_column(chunk, ["unit", "units"])
    sensor_name_col = find_column(chunk, ["sensor_name", "source", "dataset"])
    quantity_col = find_column(chunk, ["quantity", "description"])

    if dt_col is None or lat_col is None or lon_col is None:
        raise ValueError("The input CSV must contain timestamp, latitude, and longitude information.")

    # ------------------------------------------------------------------
    # Case 1: sensor-style format with one value column and a variable code.
    # ------------------------------------------------------------------
    if value_col is not None and sensor_code_col is not None:
        out = pd.DataFrame()
        out["station_id"] = chunk[station_col].astype(str).str.strip() if station_col else (
            chunk[lat_col].astype(str) + "_" + chunk[lon_col].astype(str)
        )
        out["dt"] = chunk[dt_col]
        out["lat"] = chunk[lat_col]
        out["lon"] = chunk[lon_col]
        out["raw_variable"] = chunk[sensor_code_col]
        out["variable"] = out["raw_variable"].map(map_to_standard_variable)
        out["value"] = chunk[value_col]
        out["source"] = chunk[sensor_name_col] if sensor_name_col else "HadISD"
        out["quantity"] = chunk[quantity_col] if quantity_col else np.nan
        out["unit_original"] = chunk[unit_col] if unit_col else np.nan

        out = out[out["variable"].notna()].copy()
        if len(out) == 0:
            return out

        # Convert values variable by variable.
        converted_parts = []
        for var, part in out.groupby("variable"):
            part = part.copy()
            part["value"] = convert_units(var, part["value"])
            part["unit"] = VARIABLES[var]["unit"]
            converted_parts.append(part)
        return pd.concat(converted_parts, ignore_index=True) if converted_parts else out

    # ------------------------------------------------------------------
    # Case 2: wide format with msl/u10/v10/t2m columns.
    # ------------------------------------------------------------------
    wide_parts = []
    lower_cols = {str(c).lower(): c for c in chunk.columns}

    for standard_var, meta in VARIABLES.items():
        raw_col = None
        for raw_name in meta["raw_names"] + [standard_var.lower(), standard_var.upper()]:
            if str(raw_name).lower() in lower_cols:
                raw_col = lower_cols[str(raw_name).lower()]
                break
        if raw_col is None:
            continue

        part = pd.DataFrame()
        part["station_id"] = chunk[station_col].astype(str).str.strip() if station_col else (
            chunk[lat_col].astype(str) + "_" + chunk[lon_col].astype(str)
        )
        part["dt"] = chunk[dt_col]
        part["lat"] = chunk[lat_col]
        part["lon"] = chunk[lon_col]
        part["raw_variable"] = raw_col
        part["variable"] = standard_var
        part["value"] = convert_units(standard_var, chunk[raw_col])
        part["unit"] = meta["unit"]
        part["source"] = "HadISD"
        part["quantity"] = meta["description"]
        part["unit_original"] = np.nan
        wide_parts.append(part)

    if not wide_parts:
        raise ValueError(
            "No recognised HadISD variables found. Expected sensor_code values or columns such as msl, u10, v10, t2m."
        )

    return pd.concat(wide_parts, ignore_index=True)

## 3. Chunk-based cleaning and quality control for all variables

The cleaning logic is applied consistently to every available standard variable:

1. parse timestamps,
2. convert values and coordinates to numeric values,
3. remove invalid essential fields,
4. remove invalid coordinates,
5. keep only observations from 2024,
6. apply a variable-specific physical plausibility range,
7. remove exact duplicate station-time-variable records,
8. export a cleaned all-variable table and summary statistics.

In [ ]:
def make_clean_chunk(chunk: pd.DataFrame):
    long_df = harmonise_chunk_to_long(chunk)
    raw_selected_rows = len(long_df)

    if raw_selected_rows == 0:
        return long_df, long_df, {"selected_rows": 0, "clean_rows": 0}

    df = long_df.copy()
    df["station_id"] = df["station_id"].astype(str).str.strip()
    df["dt"] = pd.to_datetime(df["dt"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

    df["flag_missing_station_id"] = df["station_id"].isna() | (df["station_id"] == "") | (df["station_id"].str.lower() == "nan")
    df["flag_invalid_datetime"] = df["dt"].isna()
    df["flag_invalid_value"] = df["value"].isna()
    df["flag_invalid_coordinates"] = (
        df["lat"].isna() | df["lon"].isna() |
        ~df["lat"].between(-90, 90) |
        ~df["lon"].between(-180, 180)
    )
    df["flag_not_target_year"] = df["dt"].dt.year.ne(YEAR)

    # Variable-specific physical plausibility check.
    df["flag_out_of_physical_range"] = False
    for var, meta in VARIABLES.items():
        vmin, vmax = meta["range"]
        mask = df["variable"].eq(var)
        df.loc[mask, "flag_out_of_physical_range"] = ~df.loc[mask, "value"].between(vmin, vmax)

    before_dup = len(df)
    df = df.drop_duplicates(subset=["station_id", "dt", "variable"])
    duplicates_removed = before_dup - len(df)

    clean_mask = ~(
        df["flag_missing_station_id"] |
        df["flag_invalid_datetime"] |
        df["flag_invalid_value"] |
        df["flag_invalid_coordinates"] |
        df["flag_not_target_year"] |
        df["flag_out_of_physical_range"]
    )
    clean = df.loc[clean_mask].copy()
    clean["date"] = clean["dt"].dt.date
    clean["month"] = clean["dt"].dt.month

    keep_cols = [
        "station_id", "dt", "date", "month", "variable", "raw_variable", "value", "unit",
        "lat", "lon", "source", "quantity", "unit_original"
    ]
    keep_cols = [c for c in keep_cols if c in clean.columns]
    clean = clean[keep_cols]

    summary = {
        "selected_rows": raw_selected_rows,
        "duplicates_removed": duplicates_removed,
        "clean_rows": len(clean),
        "missing_station_id": int(df["flag_missing_station_id"].sum()),
        "invalid_datetime": int(df["flag_invalid_datetime"].sum()),
        "invalid_value": int(df["flag_invalid_value"].sum()),
        "invalid_coordinates": int(df["flag_invalid_coordinates"].sum()),
        "not_target_year": int(df["flag_not_target_year"].sum()),
        "out_of_physical_range": int(df["flag_out_of_physical_range"].sum()),
    }

    # Variable-level counts for this chunk.
    for var in VARIABLES:
        var_mask = df["variable"].eq(var)
        summary[f"selected_{var}"] = int(var_mask.sum())
        summary[f"clean_{var}"] = int(clean["variable"].eq(var).sum())
        summary[f"out_of_range_{var}"] = int((var_mask & df["flag_out_of_physical_range"]).sum())

    return clean, df, summary

In [ ]:
# Main cleaning pass for all variables
summary_totals = defaultdict(int)
variable_counts = defaultdict(int)
monthly_counts = defaultdict(int)
daily_sum = defaultdict(float)
daily_count = defaultdict(int)
station_stats = defaultdict(lambda: {
    "n_obs": 0,
    "lat_sum": 0.0,
    "lon_sum": 0.0,
    "lat_count": 0,
    "lon_count": 0,
    "min_dt": None,
    "max_dt": None,
    "value_sum": 0.0,
    "value_count": 0,
    "value_min": np.inf,
    "value_max": -np.inf,
})
station_variable_counts = defaultdict(int)
station_month_counts = defaultdict(int)
plot_samples_by_variable = defaultdict(list)

if CLEANED_CSV.exists():
    CLEANED_CSV.unlink()

first_write = True
chunk_counter = 0

for chunk in pd.read_csv(CSV_PATH, chunksize=CHUNKSIZE, low_memory=False):
    chunk_counter += 1
    clean, flagged, summary = make_clean_chunk(chunk)

    for k, v in summary.items():
        summary_totals[k] += int(v)

    if len(clean) > 0:
        clean.to_csv(CLEANED_CSV, mode="w" if first_write else "a", header=first_write, index=False)
        first_write = False

        for var, n in clean.groupby("variable").size().items():
            variable_counts[var] += int(n)

        for (var, m), n in clean.groupby(["variable", "month"]).size().items():
            monthly_counts[(var, int(m))] += int(n)

        for (var, d), row in clean.groupby(["variable", "date"])["value"].agg(["sum", "count"]).iterrows():
            daily_sum[(var, d)] += float(row["sum"])
            daily_count[(var, d)] += int(row["count"])

        grouped = clean.groupby(["station_id", "variable"])
        for (sid, var), part in grouped:
            key = (str(sid), str(var))
            st = station_stats[key]
            st["n_obs"] += len(part)
            st["lat_sum"] += part["lat"].sum(skipna=True)
            st["lon_sum"] += part["lon"].sum(skipna=True)
            st["lat_count"] += part["lat"].notna().sum()
            st["lon_count"] += part["lon"].notna().sum()
            st["value_sum"] += part["value"].sum(skipna=True)
            st["value_count"] += part["value"].notna().sum()
            st["value_min"] = min(st["value_min"], part["value"].min(skipna=True))
            st["value_max"] = max(st["value_max"], part["value"].max(skipna=True))

            mn = part["dt"].min()
            mx = part["dt"].max()
            st["min_dt"] = mn if st["min_dt"] is None or mn < st["min_dt"] else st["min_dt"]
            st["max_dt"] = mx if st["max_dt"] is None or mx > st["max_dt"] else st["max_dt"]

        for (sid, var), n in clean.groupby(["station_id", "variable"]).size().items():
            station_variable_counts[(str(sid), str(var))] += int(n)

        for (sid, var, m), n in clean.groupby(["station_id", "variable", "month"]).size().items():
            station_month_counts[(str(sid), str(var), int(m))] += int(n)

        # Keep samples for plotting by variable.
        for var, part in clean.groupby("variable"):
            current = sum(len(x) for x in plot_samples_by_variable[var])
            if current < MAX_SAMPLE_ROWS_PER_VARIABLE:
                take_n = min(len(part), MAX_SAMPLE_ROWS_PER_VARIABLE - current)
                plot_samples_by_variable[var].append(part.sample(n=take_n, random_state=42) if len(part) > take_n else part.copy())

    if chunk_counter % 5 == 0:
        print(f"Processed {chunk_counter} chunks...")

print("Processed chunks:", chunk_counter)
print("Cleaned all-variable CSV saved to:", CLEANED_CSV)
print("Summary totals:")
for k, v in dict(summary_totals).items():
    print(f"  {k}: {v}")

## 4. Save all-variable summary tables

In [ ]:
# Global cleaning summary
summary_df = pd.DataFrame([dict(summary_totals)])
summary_df["input_file"] = CSV_PATH.name
summary_df["year"] = YEAR
summary_df.to_csv(SUMMARY_CSV, index=False)
display(summary_df.T.rename(columns={0: "value"}))

# Variable summary
variable_summary_rows = []
for var, meta in VARIABLES.items():
    variable_summary_rows.append({
        "variable": var,
        "description": meta["description"],
        "unit": meta["unit"],
        "expected_min": meta["range"][0],
        "expected_max": meta["range"][1],
        "selected_rows": summary_totals.get(f"selected_{var}", 0),
        "clean_rows": summary_totals.get(f"clean_{var}", 0),
        "out_of_range_rows": summary_totals.get(f"out_of_range_{var}", 0),
    })
variable_summary = pd.DataFrame(variable_summary_rows)
variable_summary.to_csv(VARIABLE_SUMMARY_CSV, index=False)
display(variable_summary)

# Monthly summary by variable
monthly_summary_rows = []
for var in VARIABLES:
    for m in range(1, 13):
        monthly_summary_rows.append({
            "variable": var,
            "month": m,
            "month_name": pd.to_datetime(m, format="%m").month_name(),
            "n_observations": monthly_counts.get((var, m), 0),
        })
monthly_summary = pd.DataFrame(monthly_summary_rows)
monthly_summary.to_csv(MONTHLY_SUMMARY_CSV, index=False)
display(monthly_summary.head(20))

# Station summary by variable
station_rows = []
for (sid, var), st in station_stats.items():
    n_obs = st["n_obs"]
    station_rows.append({
        "station_id": sid,
        "variable": var,
        "n_observations": n_obs,
        "completeness_pct": 100 * n_obs / EXPECTED_HOURLY_OBS,
        "lat": st["lat_sum"] / st["lat_count"] if st["lat_count"] else np.nan,
        "lon": st["lon_sum"] / st["lon_count"] if st["lon_count"] else np.nan,
        "first_timestamp": st["min_dt"],
        "last_timestamp": st["max_dt"],
        "mean_value": st["value_sum"] / st["value_count"] if st["value_count"] else np.nan,
        "min_value": st["value_min"] if np.isfinite(st["value_min"]) else np.nan,
        "max_value": st["value_max"] if np.isfinite(st["value_max"]) else np.nan,
    })
station_summary = pd.DataFrame(station_rows).sort_values(["variable", "n_observations"], ascending=[True, False])
station_summary.to_csv(STATION_SUMMARY_CSV, index=False)
display(station_summary.head(20))

# Station-month availability table by variable
station_month_df = pd.DataFrame(
    [(sid, var, m, n) for (sid, var, m), n in station_month_counts.items()],
    columns=["station_id", "variable", "month", "n_observations"]
)
station_month_df.to_csv(STATION_MONTH_CSV, index=False)
display(station_month_df.head(20))

## 5. Temporal gap analysis for all variables

This step computes the maximum time gap between consecutive observations for each station and each variable.

In [ ]:
def compute_max_gaps(cleaned_csv: Path) -> pd.DataFrame:
    if not cleaned_csv.exists() or cleaned_csv.stat().st_size == 0:
        return pd.DataFrame(columns=["station_id", "variable", "max_gap_hours", "median_gap_hours"])

    gap_df = pd.read_csv(cleaned_csv, usecols=["station_id", "variable", "dt"], parse_dates=["dt"])
    gap_df = gap_df.dropna(subset=["station_id", "variable", "dt"])
    gap_df["station_id"] = gap_df["station_id"].astype(str)
    gap_df["variable"] = gap_df["variable"].astype(str)
    gap_df = gap_df.sort_values(["station_id", "variable", "dt"])
    gap_df["gap_hours"] = gap_df.groupby(["station_id", "variable"])["dt"].diff().dt.total_seconds() / 3600

    return gap_df.groupby(["station_id", "variable"])["gap_hours"].agg(
        max_gap_hours="max",
        median_gap_hours="median"
    ).reset_index()

try:
    gap_summary = compute_max_gaps(CLEANED_CSV)
    gap_summary.to_csv(MAX_GAP_CSV, index=False)
    display(gap_summary.sort_values("max_gap_hours", ascending=False).head(20))
except MemoryError:
    gap_summary = pd.DataFrame(columns=["station_id", "variable", "max_gap_hours", "median_gap_hours"])
    print("MemoryError: gap analysis skipped. The other cleaning outputs are still valid.")

print("Gap summary saved to:", MAX_GAP_CSV)

## 6. Generate report-ready figures

The QC analysis above is applied to all variables. To keep the report compact, the diagnostic plots below focus mainly on pressure (`PRESS`) as a representative variable. The same summary tables exist for the other variables.

In [ ]:
def save_current_fig(filename: str):
    path = FIG_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    print("Saved:", path)

# Plot samples by variable
sample_by_variable = {}
for var, parts in plot_samples_by_variable.items():
    sample_by_variable[var] = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()

# Build station coordinates using all variables.
if len(station_summary) > 0:
    station_coords = station_summary.groupby("station_id").agg(
        lat=("lat", "mean"),
        lon=("lon", "mean"),
        n_variables=("variable", "nunique"),
        n_observations=("n_observations", "sum"),
    ).reset_index()
else:
    station_coords = pd.DataFrame(columns=["station_id", "lat", "lon", "n_variables", "n_observations"])

# 00 Cleaned observation counts by variable
plt.figure(figsize=(8, 5))
plt.bar(variable_summary["variable"], variable_summary["clean_rows"])
plt.xlabel("Variable")
plt.ylabel("Number of cleaned observations")
plt.title("Cleaned HadISD observations by variable - 2024")
plt.grid(axis="y", alpha=0.3)
save_current_fig("00_cleaned_observations_by_variable.png")

In [ ]:
# 01 Map-like station coverage figure with approximate Adriatic coastline guide
if len(station_coords) > 0:
    fig, ax = plt.subplots(figsize=(8, 7))

    # Approximate guide lines for the two Adriatic coastlines.
    # These are not used for analysis; they only make the station-location plot easier to read in the report.
    italy_lon = [12.3, 12.35, 12.25, 12.20, 12.55, 13.50, 14.22, 15.20, 16.87, 17.94, 18.49]
    italy_lat = [45.5, 45.2, 44.8, 44.4, 44.0, 43.6, 42.5, 41.9, 41.1, 40.6, 40.2]
    east_lon = [13.75, 13.85, 14.30, 15.23, 16.44, 17.20, 18.09, 18.80, 19.45, 19.50]
    east_lat = [45.6, 44.9, 44.5, 44.1, 43.5, 43.0, 42.65, 42.0, 41.3, 40.5]

    ax.plot(italy_lon, italy_lat, linewidth=1.5, label="Approximate Adriatic coastline")
    ax.plot(east_lon, east_lat, linewidth=1.5)
    scatter = ax.scatter(
        station_coords["lon"], station_coords["lat"],
        s=35, alpha=0.85, edgecolors="black", linewidths=0.3
    )

    ax.set_xlim(11.5, 20.5)
    ax.set_ylim(39.0, 46.5)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title("HadISD 2024 station coverage over the Adriatic domain")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower left")
    save_current_fig("01_hadisd_station_map.png")

In [ ]:
# Pressure-focused diagnostic figures for the report
PLOT_VARIABLE = "PRESS"
pressure_sample = sample_by_variable.get(PLOT_VARIABLE, pd.DataFrame())

# Monthly observation counts for pressure
press_monthly = monthly_summary[monthly_summary["variable"] == PLOT_VARIABLE].copy()
plt.figure(figsize=(10, 5))
plt.bar(press_monthly["month_name"], press_monthly["n_observations"])
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of cleaned observations")
plt.title("Monthly HadISD pressure observation counts - 2024")
plt.grid(axis="y", alpha=0.3)
save_current_fig("02_monthly_observation_counts_PRESS.png")

# Station-month availability heatmap for pressure
press_station_summary = station_summary[station_summary["variable"] == PLOT_VARIABLE].copy()
press_station_month = station_month_df[station_month_df["variable"] == PLOT_VARIABLE].copy()
if len(press_station_month) > 0:
    press_matrix = press_station_month.pivot_table(
        index="station_id", columns="month", values="n_observations", fill_value=0
    ).reindex(columns=list(range(1, 13)), fill_value=0)
    top_stations = press_station_summary.head(50)["station_id"].astype(str).tolist()
    heat = press_matrix.reindex(top_stations).fillna(0)
    plt.figure(figsize=(10, max(6, 0.18 * len(heat))))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label="Number of observations")
    plt.xticks(ticks=np.arange(12), labels=[str(m) for m in range(1, 13)])
    plt.yticks(ticks=np.arange(len(heat.index)), labels=heat.index)
    plt.xlabel("Month")
    plt.ylabel("Station ID")
    plt.title("Station-month availability heatmap for HadISD pressure - 2024")
    save_current_fig("03_station_month_availability_heatmap_PRESS.png")

# Pressure distribution
if len(pressure_sample) > 0:
    plt.figure(figsize=(9, 5))
    plt.hist(pressure_sample["value"].dropna(), bins=60)
    plt.xlabel("Pressure (hPa)")
    plt.ylabel("Frequency")
    plt.title("Distribution of cleaned HadISD pressure observations")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("04_pressure_distribution.png")

# Pressure boxplot by month
if len(pressure_sample) > 0:
    box_data = [pressure_sample.loc[pressure_sample["month"] == m, "value"].dropna().values for m in range(1, 13)]
    plt.figure(figsize=(10, 5))
    plt.boxplot(box_data, labels=[str(m) for m in range(1, 13)], showfliers=False)
    plt.xlabel("Month")
    plt.ylabel("Pressure (hPa)")
    plt.title("Monthly pressure distribution after cleaning")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("05_pressure_boxplot_by_month.png")

# Daily mean pressure time series
press_daily_rows = []
for (var, d), s in daily_sum.items():
    if var == PLOT_VARIABLE:
        press_daily_rows.append({
            "date": d,
            "daily_mean": s / daily_count[(var, d)] if daily_count[(var, d)] else np.nan,
        })
press_daily = pd.DataFrame(press_daily_rows)
if len(press_daily) > 0:
    press_daily["date"] = pd.to_datetime(press_daily["date"])
    press_daily = press_daily.sort_values("date")
    plt.figure(figsize=(12, 5))
    plt.plot(press_daily["date"], press_daily["daily_mean"])
    plt.xlabel("Date")
    plt.ylabel("Daily mean pressure (hPa)")
    plt.title("Daily mean HadISD pressure during 2024")
    plt.grid(True, alpha=0.3)
    save_current_fig("06_daily_mean_pressure_timeseries.png")

# Station completeness distribution for pressure
if len(press_station_summary) > 0:
    plt.figure(figsize=(9, 5))
    plt.hist(press_station_summary["completeness_pct"].dropna(), bins=30)
    plt.xlabel("Completeness (%)")
    plt.ylabel("Number of stations")
    plt.title("Station-level completeness distribution for pressure")
    plt.grid(axis="y", alpha=0.3)
    save_current_fig("07_station_completeness_distribution_PRESS.png")

# Maximum temporal gap for pressure
if len(gap_summary) > 0:
    press_gap = gap_summary[gap_summary["variable"] == PLOT_VARIABLE].copy()
    if len(press_gap) > 0:
        top_gap = press_gap.sort_values("max_gap_hours", ascending=False).head(20)
        plt.figure(figsize=(10, 5))
        plt.bar(top_gap["station_id"].astype(str), top_gap["max_gap_hours"])
        plt.xticks(rotation=60, ha="right")
        plt.ylabel("Maximum gap (hours)")
        plt.title("Largest temporal gaps in HadISD pressure records")
        plt.grid(axis="y", alpha=0.3)
        save_current_fig("09_max_temporal_gap_PRESS_by_station.png")

## 7. Final output summary

In [ ]:
print("Notebook completed successfully.")
print("
Main outputs:")
print("Cleaned all-variable CSV:", CLEANED_CSV)
print("Cleaning summary:", SUMMARY_CSV)
print("Variable summary:", VARIABLE_SUMMARY_CSV)
print("Station summary:", STATION_SUMMARY_CSV)
print("Monthly summary:", MONTHLY_SUMMARY_CSV)
print("Station-month availability:", STATION_MONTH_CSV)
print("Max temporal gaps:", MAX_GAP_CSV)
print("Figures folder:", FIG_DIR)

print("
Important note:")
print("The raw CSV and cleaned CSV can be large. Do not push them to GitHub unless the team explicitly wants them.")
print("For the report, usually push/use the notebook, summary tables, and PNG figures only.")

## Report interpretation

This notebook supports the HadISD report section by making the quality-control workflow more consistent with the full 2024 consolidated file.

Recommended wording for the report:

> The quality-control workflow was applied to all variables available in the HadISD 2024 consolidated file, including atmospheric pressure, 10 m wind components, and 2 m air temperature. Since the report has limited space, the diagnostic plots are shown mainly for pressure, which was selected as a representative variable because it can later be directly compared with ERA5 mean sea-level pressure. The same cleaning logic, including timestamp validation, coordinate checks, missing-value detection, duplicate removal, and physical plausibility checks, was applied consistently to the remaining variables.

Recommended figure captions:

- **Figure X. Spatial distribution of HadISD stations over the Adriatic domain.** The station locations are displayed on a geographic map to verify that the selected observations fall within the project area and to assess the spatial coverage of the available HadISD network.
- **Figure X. Station-month availability heatmap for HadISD pressure observations in 2024.** The heatmap is shown for pressure as a representative variable and highlights missing-data patterns and incomplete station records.
- **Figure X. Distribution of cleaned HadISD pressure values.** This plot verifies the physical plausibility of the pressure observations after the all-variable cleaning workflow.
- **Figure X. Daily mean HadISD pressure during 2024.** The time series summarises the temporal evolution of the cleaned pressure observations.
- **Figure X. Maximum temporal gap by station for HadISD pressure observations.** This plot identifies stations with long interruptions in the pressure time series.